# Oracle Feature Ablation Study

Ablates feature families one at a time and in combination to measure each family's contribution to oracle performance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import pickle as pkl
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

print('Libraries loaded.')

## Section 1 — Load All Radiomics Features from pkl Files

In [ ]:
RADIOMICS_PATH = '../../Results/Analysis_Results/Radiomics/Tumor_WT/'
analysis_types = sorted([f.replace('.pkl', '') for f in os.listdir(RADIOMICS_PATH) if f.endswith('.pkl')])
print('Analysis types:', analysis_types)

def load_all_radiomics(path, types):
    frames = []
    for atype in types:
        fpath = os.path.join(path, f'{atype}.pkl')
        with open(fpath, 'rb') as f:
            raw = pkl.load(f)
        n_feats = 0
        for prop_name, patient_dict in raw.items():
            df = pd.DataFrame.from_dict(patient_dict, orient='index').astype(float)
            df.columns = [f'{prop_name}_{col}_{atype}' for col in df.columns]
            frames.append(df)
            n_feats += df.shape[1]
        print(f'  {atype:15s}: {n_feats} features')
    combined = frames[0]
    for df in frames[1:]:
        combined = combined.join(df, how='outer')
    print(f'Total radiomics: {combined.shape[1]}')
    return combined

radiomics_df = load_all_radiomics(RADIOMICS_PATH, analysis_types)
print(f'Radiomics matrix: {radiomics_df.shape}')

## Section 2 — Load Model-Based Features

In [ ]:
volume_features   = ['ED', 'ET', 'NCR', 'WT_volume', 'TC_volume', 'ET_volume', 'TC_WT_ratio', 'ET_WT_ratio', 'ET_TC_ratio']
curvature_features = ['mean_gaussian_curvature', 'std_gaussian_curvature', 'pos', 'neg', 'pos_count', 'neg_count']

volume_df = pd.read_csv('../../Results/Analysis_Results/volume/GLI-Tumor_volumns.csv', index_col='Unnamed: 0')[volume_features]
prob_df   = pd.read_csv('../../Results/Analysis_Results/probability/Probability_Tumor_boundary.csv', index_col='Unnamed: 0')
curv_df   = pd.read_csv('../../Results/Analysis_Results/curverature/curverature.csv', index_col='Unnamed: 0')[curvature_features]
sal_df    = pd.read_csv('../../Results/Analysis_Results/Saliency/Saliency.csv', index_col='Unnamed: 0')

model_based_df = volume_df.join(prob_df, how='outer').join(curv_df, how='outer').join(sal_df, how='outer')
model_based_cols = list(model_based_df.columns)

print(f'Volume:      {len(volume_features)}')
print(f'Probability: {prob_df.shape[1]}')
print(f'Curvature:   {len(curvature_features)}')
print(f'Saliency:    {sal_df.shape[1]}')
print(f'Total model-based: {model_based_df.shape[1]}')

## Section 3 — Load All 3 Segmentation Model Performances

In [ ]:
def load_unet_result(path):
    df = pd.read_csv(path, index_col='Unnamed: 0')
    df.index = [idx.split('-seg')[0]  # strip '-seg' suffix from U-Net inference output filenames for idx in df.index]
    df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard']  # Dice is the primary metric; Jaccard unused, axis=1, inplace=True)
    return df

def load_nnunet_result(path):
    with open(path, 'r') as f:
        data = json.load(f)
    rows = []
    for case in data['metric_per_case']:
        name = case['reference_file'].split('/')[-1].split('.')[0]
        rows.append({'index': name, 'WT dice': case['metrics']['(2, 1, 3)']['Dice'],
                     'TC dice': case['metrics']['(2, 3)']['Dice'], 'ET dice': case['metrics']['(3,)']['Dice']})
    return pd.DataFrame(rows).set_index('index')

def load_transbts_result(path):
    with open(path, 'r') as f:
        data = json.load(f)
    rows = []
    for case_id, case in data.items():
        rows.append({'index': case_id, 'WT dice': case['WT'][0], 'TC dice': case['TC'][0], 'ET dice': case['ET'][0]})
    return pd.DataFrame(rows).set_index('index')

unet_df     = load_unet_result('../../Results/Result/Vanilla_Unet/Unet_test_dice.csv')
nnunet_df   = load_nnunet_result('../../Results/Result/nnUnet/nnUNetTrainerNoDA/summary.json')
transbts_df = load_transbts_result('../../Results/Result/TransBTS/submission/TransBTS2023-11-03/TransBTS_summary.json')
print(f'U-Net: {unet_df.shape}, nnU-Net: {nnunet_df.shape}, TransBTS: {transbts_df.shape}')

## Section 4 — Build Full Feature Matrix & Significance Subsets

In [ ]:
X_full = radiomics_df.join(model_based_df, how='inner').dropna()
print(f'Full feature matrix (after dropna): {X_full.shape}')

summary_df = pd.read_csv('../../Results/Analysis_Results/Radiomics/summary/Tumor_WT_summary.csv')
summary_df['Name'] = summary_df['Feature'] + '_' + summary_df['Modality'] + '_' + summary_df['Method']

sig_names   = set(summary_df[summary_df['P-value'] == '***']['Name'].unique())
large_names = set(summary_df[(summary_df['P-value'] == '***') & (summary_df['Effect_Size'] == 'Large')]['Name'].unique())

radiomics_cols_in_X = [c for c in X_full.columns if c not in model_based_cols]
model_cols_in_X     = [c for c in model_based_cols if c in X_full.columns]

all_cols   = list(X_full.columns)
sig_cols   = [c for c in radiomics_cols_in_X if c in sig_names]   + model_cols_in_X
large_cols = [c for c in radiomics_cols_in_X if c in large_names] + model_cols_in_X

print(f'A: All features:          {len(all_cols)}')
print(f'   (radiomics: {len(radiomics_cols_in_X)}, model-based: {len(model_cols_in_X)})')
print(f'B: Significant (p<0.05):  {len(sig_cols)}')
print(f'   (radiomics: {len(sig_cols)-len(model_cols_in_X)}, model-based: {len(model_cols_in_X)})')
print(f'C: Large effect only:     {len(large_cols)}')
print(f'   (radiomics: {len(large_cols)-len(model_cols_in_X)}, model-based: {len(model_cols_in_X)})')

## Section 5 — Ablation: 3 Models × 3 Feature Subsets

In [ ]:
def run_gbr_cv(X, y, n_splits=5, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    mae_scores, r2_scores, mse_scores = [], [], []
    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_train)
        X_te = scaler.transform(X_test)
        model = GradientBoostingRegressor(n_estimators=150,  # tuned via Bayesian HPO (see Oracle_Model_with_HPO.ipynb), learning_rate=0.1, max_depth=5, subsample=0.75, random_state=42)
        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        mae_scores.append(mean_absolute_error(y_test, y_pred))
        r2_scores.append(r2_score(y_test, y_pred))
        mse_scores.append(mean_squared_error(y_test, y_pred))
    return {'MAE': (np.mean(mae_scores), np.std(mae_scores)),
            'R2':  (np.mean(r2_scores),  np.std(r2_scores)),
            'MSE': (np.mean(mse_scores), np.std(mse_scores))}

MODELS = {'Vanilla U-Net': unet_df, 'nnU-Net': nnunet_df, 'TransBTS': transbts_df}
SUBSETS = {f'A: All ({len(all_cols)})': all_cols, f'B: Sig ({len(sig_cols)})': sig_cols, f'C: Large ({len(large_cols)})': large_cols}

all_results = {}
print('Running ablation (5-fold CV)...\n')
for model_name, perf_df in MODELS.items():
    common_idx = X_full.index.intersection(perf_df.index)
    y = perf_df.loc[common_idx, 'WT dice']
    model_results = {}
    print(f'{model_name}:')
    for label, cols in SUBSETS.items():
        X_sub = X_full.loc[common_idx, [c for c in cols if c in X_full.columns]]
        res = run_gbr_cv(X_sub, y)
        model_results[label] = res
        print(f'  {label:25s}  MAE={res["MAE"][0]:.3f}\u00b1{res["MAE"][1]:.3f}  MSE={res["MSE"][0]:.4f}\u00b1{res["MSE"][1]:.4f}  R2={res["R2"][0]:.3f}\u00b1{res["R2"][1]:.3f}')
    all_results[model_name] = model_results
    print()

## Section 6 — Figure

In [ ]:
model_names  = list(MODELS.keys())
subset_keys  = list(SUBSETS.keys())
subset_short = ['All', 'Sig', 'Large']
colors       = ['lightblue', 'lightcoral', '#b0d4b0']
x = np.arange(len(subset_keys))

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey='row')
fig.suptitle('Oracle Feature Subset Ablation — All 3 Segmentation Models\n(GBR, 5-fold CV, WT Dice target)', fontsize=12, fontweight='bold', y=1.02)

for col, model_name in enumerate(model_names):
    for row, metric in enumerate(['MAE', 'R2']):
        ax = axes[row, col]
        means = [all_results[model_name][sk][metric][0] for sk in subset_keys]
        stds  = [all_results[model_name][sk][metric][1] for sk in subset_keys]
        ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor='white', width=0.55)
        ax.set_xticks(x)
        ax.set_xticklabels(subset_short, fontsize=10)
        for xi, (m, s) in enumerate(zip(means, stds)):
            offset = s + 0.002 if metric == 'MAE' else s + 0.005
            ax.text(xi, m + offset, f'{m:.3f}', ha='center', va='bottom', fontsize=8)
        if row == 0:
            ax.set_title(model_name, fontsize=11, fontweight='bold')
        if col == 0:
            ax.set_ylabel('MAE (lower=better)' if row==0 else 'R\u00b2 (higher=better)', fontsize=10)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

patches = [mpatches.Patch(color=c, label=s) for c,s in zip(colors, subset_short)]
fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5,-0.05))
plt.tight_layout()
out_path = '../../Results/Figures/oracle_ablation_detailed.pdf'
plt.savefig(out_path, bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved: {out_path}')

## Section 7 — Save JSON

In [ ]:
output = {
    'feature_subsets': {
        'A_all':   {'n': len(all_cols), 'radiomics': len(radiomics_cols_in_X), 'model_based': len(model_cols_in_X)},
        'B_sig':   {'n': len(sig_cols), 'radiomics': len(sig_cols)-len(model_cols_in_X), 'model_based': len(model_cols_in_X)},
        'C_large': {'n': len(large_cols), 'radiomics': len(large_cols)-len(model_cols_in_X), 'model_based': len(model_cols_in_X)},
    },
    'results': {
        model_name: {
            label: {'MAE_mean': round(res['MAE'][0],4), 'MAE_std': round(res['MAE'][1],4),
                    'MSE_mean': round(res['MSE'][0],4), 'MSE_std': round(res['MSE'][1],4),
                    'R2_mean':  round(res['R2'][0],4),  'R2_std':  round(res['R2'][1],4)}
            for label, res in model_results.items()
        }
        for model_name, model_results in all_results.items()
    }
}
out_json = '../../Results/Json_summary/oracle_ablation_detailed.json'
with open(out_json, 'w') as f:
    json.dump(output, f, indent=4)
print(f'Saved: {out_json}')